In [6]:
# -*- coding: utf-8 -*-
"""
FFT 可视化总脚本（一次读、内部循环画）
------------------------------------------------------------
包含：
- 工具：_load_pt/_to_np/_parse_info_from_name/_infer_times/_sec_str/_wrap/_desc_from_info
- 基础绘图（FFT 版）：
  1) plot_xy_compare_attack_original_avg(...)             # 单图 1×4（FFT 幅度/可选 log）
  2) plot_frames_y_and_diff_side_by_side_avg(...)         # 多帧 5×10（左：FFT(y_k)，右：FFT(y_k−u_k)）
  3) plot_rows_attack_vs_train_meanN(...)                 # 多行 8 图（多个 attack 与同一个 train）
- 只读一次、在函数内循环画的“批量版本”：
  4) plot_xy_and_frames_variants_once(...)                # attack + original 只读取一次，内部循环画（单图+多帧）
  5) plot_rows_attack_vs_train_meanN_multi_once(...)      # train 一次，attack 各一次，分别画出多行 8 图（可多个 log 版本）

说明要点：
- 所有函数先对 (H,W) 做 FFT（可选 fftshift），在复域对样本维做均值，再取幅度或 log10(幅度+eps) 来画。
- 差分图在“复域先相减再取幅度”，对应 FFT(x-a) 或 FFT(y_k - u_k) 的幅度，避免 |X|-|A| 的非线性偏差。
- “一次读内部循环画”的两个函数可以替换你外部 for 循环重复加载的做法，大幅减少 I/O。
"""

import re
import textwrap
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt


# ========== Utils ==========

def _load_pt(pt_path: str):
    """读取 .pt，必须包含 'x' 和 'y' 两个 key。"""
    data = torch.load(pt_path, map_location="cpu", weights_only=False)
    if not all(k in data for k in ["x", "y"]):
        raise KeyError(f"{pt_path} must contain 'x' and 'y'. Keys: {list(data.keys())}")
    return data

def _to_np(t):
    """Torch -> NumPy（detach+cpu）。"""
    if isinstance(t, torch.Tensor):
        return t.detach().cpu().numpy()
    return np.asarray(t)

def _parse_info_from_name(path_str):
    """尽量从文件名解析元信息；取不到的留空。"""
    name = Path(path_str).name
    g = lambda pat: (m.group(1) if (m := re.search(pat, name)) else "")
    return {
        "solver":  g(r"solver=([A-Za-z0-9_]+)"),
        "nu":      g(r"nu([0-9eE\.\-]+)"),
        "t_final": g(r"t([0-9]+(?:\.[0-9]+)?)"),
        "norm":    g(r"norm([^\W_]+)"),
        "alpha":   g(r"alpha([0-9eE\.\-]+)"),
        "epsilon": g(r"epsilon([0-9eE\.\-]+)"),
        "steps":   g(r"steps([0-9]+)"),
        "N":       g(r"N([0-9]+)"),
        "nx":      g(r"nx([0-9]+)"),
    }

def _infer_times(T, t_final_str):
    """
    a: t=0; u: t=Δt..TΔt; x: frame1 => t=Δt; y: t=Δt..TΔt
    返回 (t_x, t_frames)，单位秒；取不到则 (None, None)。
    """
    if not t_final_str:
        return None, None
    dt = float(t_final_str) / float(T)
    t_frames = [(k+1)*dt for k in range(T)]
    t_x = dt
    return t_x, t_frames

def _sec_str(t):
    return f"{t:.3g} s" if t is not None else ""

def _wrap(text, width=38):
    return textwrap.fill(text, width=width)

def _desc_from_info(info: dict):
    keys = ["solver", "nu", "t_final", "norm", "alpha", "epsilon", "steps", "nx", "N"]
    parts = [f"{k}={info[k]}" for k in keys if info.get(k)]
    return ", ".join(parts) if parts else "(meta N/A)"


# ========== 基础绘图（FFT 版） ==========

def plot_xy_compare_attack_original_avg(
    pt_attack: str,
    pt_original: str,
    N: int,
    t_idx: int = -1,
    cmap_attack: str = "viridis",
    cmap_diff: str = "coolwarm",
    cbar_mode: str = "shared",      # "shared": 左/右各 1 根；"separate": 每幅各自
    use_log: bool = False,          # 是否对幅度取 log10
    log_eps: float = 1e-12,         # log 安全项
    center_spectrum: bool = True,   # 是否 fftshift 居中 0 频
):
    """
    单图四幅：
      [0] FFT attack: x(frame 1)
      [1] FFT attack: y(frame k)
      [2] FFT diff:   x − a   （复域差分）
      [3] FFT diff:   y − u   （复域差分）
    """
    da = _load_pt(pt_attack)
    do = _load_pt(pt_original)
    x_att, y_att = da["x"], da["y"]       # x:(N,H,W), y:(N,H,W,T)
    x_org, y_org = do["x"], do["y"]       # a:(N,H,W), u:(N,H,W,T)

    if x_att.ndim != 3 or x_org.ndim != 3:
        raise ValueError(f"x must be (N,H,W). Got {tuple(x_att.shape)} and {tuple(x_org.shape)}")
    if y_att.ndim != 4 or y_org.ndim != 4:
        raise ValueError(f"y must be (N,H,W,T). Got {tuple(y_att.shape)} and {tuple(y_org.shape)}")

    # N 与帧对齐
    N_use = min(N, x_att.shape[0], x_org.shape[0], y_att.shape[0], y_org.shape[0])
    T = min(y_att.shape[3], y_org.shape[3])
    k = (t_idx % T)

    # 空间 FFT，复域均值
    Xf = torch.fft.fft2(x_att[:N_use], dim=(-2, -1))
    Af = torch.fft.fft2(x_org[:N_use], dim=(-2, -1))
    Yf = torch.fft.fft2(y_att[:N_use], dim=(1, 2))
    Uf = torch.fft.fft2(y_org[:N_use], dim=(1, 2))
    if center_spectrum:
        Xf = torch.fft.fftshift(Xf, dim=(-2, -1))
        Af = torch.fft.fftshift(Af, dim=(-2, -1))
        Yf = torch.fft.fftshift(Yf, dim=(1, 2))
        Uf = torch.fft.fftshift(Uf, dim=(1, 2))

    Xm, Am = Xf.mean(0), Af.mean(0)      # (H,W)
    Ym, Um = Yf.mean(0), Uf.mean(0)      # (H,W,T)
    Yk, Uk = Ym[..., k], Um[..., k]      # 第 k 帧

    # 复域差分（等价于 FFT(x-a)、FFT(y_k-u_k)）
    Dxa = Xm - Am
    Dyu = Yk - Uk

    def _disp(Z: torch.Tensor) -> np.ndarray:
        A = torch.abs(Z).detach().cpu().numpy()
        if use_log:
            A = np.log10(A + float(log_eps))
        return A

    x_np = _disp(Xm)
    y_np = _disp(Yk)
    xd   = _disp(Dxa)
    yd   = _disp(Dyu)

    # 时间信息
    info_att = _parse_info_from_name(pt_attack)
    info_org = _parse_info_from_name(pt_original)
    t_final_str = info_att.get("t_final") or info_org.get("t_final") or ""
    t_x, t_frames = _infer_times(T, t_final_str)
    t_k = t_frames[k] if t_frames is not None else None

    # 色域
    if cbar_mode == "shared":
        vmin_att = min(float(x_np.min()), float(y_np.min()))
        vmax_att = max(float(x_np.max()), float(y_np.max()))
        vmin_diff = min(float(xd.min()), float(yd.min()))
        vmax_diff = max(float(xd.max()), float(yd.max()))
    else:
        vmin_att = vmax_att = vmin_diff = vmax_diff = None

    # 画图
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
    tag = "FFT(log10)" if use_log else "FFT"

    axes[0].imshow(x_np, cmap=cmap_attack, vmin=vmin_att, vmax=vmax_att); axes[0].axis("off")
    axes[0].set_title(f"{tag}: attack x\n(frame 1, t={_sec_str(t_x)})")

    axes[1].imshow(y_np, cmap=cmap_attack, vmin=vmin_att, vmax=vmax_att); axes[1].axis("off")
    axes[1].set_title(f"{tag}: attack y\n(frame {k+1}, t={_sec_str(t_k)})")

    axes[2].imshow(xd, cmap=cmap_diff, vmin=vmin_diff, vmax=vmax_diff); axes[2].axis("off")
    axes[2].set_title(f"{tag}: diff x−a")

    axes[3].imshow(yd, cmap=cmap_diff, vmin=vmin_diff, vmax=vmax_diff); axes[3].axis("off")
    axes[3].set_title(f"{tag}: diff y−u")

    if cbar_mode == "shared":
        fig.colorbar(axes[1].images[0], ax=axes[:2].tolist(), fraction=0.04, pad=0.02)   # attack 组
        fig.colorbar(axes[3].images[0], ax=axes[2:].tolist(), fraction=0.04, pad=0.02)   # diff 组
    else:
        for ax in axes:
            fig.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.04)

    # 大标题
    desc_att = ", ".join([f"{k}={v}" for k, v in info_att.items() if v])
    desc_org = ", ".join([f"{k}={v}" for k, v in info_org.items() if v])
    fig.suptitle(
        f"Mean over first N={N_use} | {tag} | attack[{desc_att}] | original[{desc_org}]",
        y=1.05, fontsize=13
    )
    plt.show()
    return fig, axes


def plot_frames_y_and_diff_side_by_side_avg(
    pt_attack: str,
    pt_original: str,
    N: int,
    grid=(5, 5),                  # 左半区 cols×rows；右半区自动扩为两倍列
    cmap_attack: str = "viridis",
    cmap_diff: str = "coolwarm",
    cbar_mode: str = "shared",    # "shared": 每组 1 根；"separate": 每帧各自
    use_log: bool = False,
    log_eps: float = 1e-12,
    center_spectrum: bool = True,
):
    """
    左：FFT(y_k) 幅度（或 log 幅度）
    右：FFT(y_k − u_k) 幅度（或 log 幅度）
    """
    da = _load_pt(pt_attack)
    do = _load_pt(pt_original)
    y_att, y_org = da["y"], do["y"]     # (N,H,W,T)

    if y_att.ndim != 4 or y_org.ndim != 4:
        raise ValueError(f"y must be (N,H,W,T). Got {tuple(y_att.shape)} and {tuple(y_org.shape)}")

    N_use = min(N, y_att.shape[0], y_org.shape[0])
    T = min(y_att.shape[3], y_org.shape[3])

    rows, cols = grid
    K = rows * cols
    num_to_plot = min(T, K)

    # FFT over (H,W) & complex-mean over samples
    Yf = torch.fft.fft2(y_att[:N_use, :, :, :T], dim=(1, 2))
    Uf = torch.fft.fft2(y_org[:N_use, :, :, :T], dim=(1, 2))
    if center_spectrum:
        Yf = torch.fft.fftshift(Yf, dim=(1, 2))
        Uf = torch.fft.fftshift(Uf, dim=(1, 2))

    Yf_m = Yf.mean(0)   # (H,W,T)
    Uf_m = Uf.mean(0)   # (H,W,T)
    Df   = Yf_m - Uf_m  # (H,W,T)

    def _disp(Z):
        A = torch.abs(Z).detach().cpu().numpy()
        if use_log:
            A = np.log10(A + float(log_eps))
        return A

    y_p = _disp(Yf_m)[..., :num_to_plot]   # 左组
    y_d = _disp(Df)[..., :num_to_plot]     # 右组

    # 时间标签
    info_att = _parse_info_from_name(pt_attack)
    info_org = _parse_info_from_name(pt_original)
    t_final_str = info_att.get("t_final") or info_org.get("t_final") or ""
    _, t_frames = _infer_times(T, t_final_str)

    # 色域
    if cbar_mode == "shared":
        vmin_left,  vmax_left  = float(np.min(y_p)), float(np.max(y_p))
        vmin_right, vmax_right = float(np.min(y_d)), float(np.max(y_d))
    else:
        vmin_left = vmax_left = vmin_right = vmax_right = None

    tag = "FFT(log10)" if use_log else "FFT"

    # 布局
    fig, axes = plt.subplots(rows, 2*cols, figsize=(2*cols*2.2, rows*2.2), constrained_layout=True)
    axes = np.asarray(axes)

    ims_left, ims_right = [], []

    # 左侧：attack y(k)
    for kk in range(K):
        r, c = divmod(kk, cols)
        ax = axes[r, c]
        if kk < num_to_plot:
            if cbar_mode == "shared":
                im = ax.imshow(y_p[..., kk], cmap=cmap_attack, vmin=vmin_left, vmax=vmax_left)
            else:
                im = ax.imshow(y_p[..., kk], cmap=cmap_attack)
            t_k = (t_frames[kk] if (t_frames is not None and kk < len(t_frames)) else None)
            ax.set_title(f"{tag}: attack y(k={kk+1})\n" + (f"t={_sec_str(t_k)}" if t_k is not None else ""), fontsize=9)
            ims_left.append(im)
        ax.axis("off")

    # 右侧：diff y(k) − u(k)
    for kk in range(K):
        r, c = divmod(kk, cols)
        ax = axes[r, cols + c]
        if kk < num_to_plot:
            if cbar_mode == "shared":
                im = ax.imshow(y_d[..., kk], cmap=cmap_diff, vmin=vmin_right, vmax=vmax_right)
            else:
                im = ax.imshow(y_d[..., kk], cmap=cmap_diff)
            t_k = (t_frames[kk] if (t_frames is not None and kk < len(t_frames)) else None)
            ax.set_title(f"{tag}: diff y(k) − u(k)\n" + (f"t={_sec_str(t_k)}" if t_k is not None else ""), fontsize=9)
            ims_right.append(im)
        ax.axis("off")

    # 色条
    if cbar_mode == "shared":
        if ims_left:
            fig.colorbar(ims_left[-1],  ax=axes[:, :cols].ravel().tolist(), fraction=0.02, pad=0.01)
        if ims_right:
            fig.colorbar(ims_right[-1], ax=axes[:, cols:].ravel().tolist(), fraction=0.02, pad=0.01)
    else:
        for kk in range(num_to_plot):
            r, c = divmod(kk, cols)
            if axes[r, c].images:
                fig.colorbar(axes[r, c].images[0], ax=axes[r, c], fraction=0.036, pad=0.01)
            if axes[r, cols + c].images:
                fig.colorbar(axes[r, cols + c].images[0], ax=axes[r, cols + c], fraction=0.036, pad=0.01)

    # 大标题
    desc_att = ", ".join([f"{k}={v}" for k, v in info_att.items() if v])
    desc_org = ", ".join([f"{k}={v}" for k, v in info_org.items() if v])
    fig.suptitle(
        f"Mean over first N={N_use} | {tag} amplitude | attack y(k) vs diff(y(k)−u(k)) | "
        f"attack[{desc_att}] | original[{desc_org}]",
        y=1.05, fontsize=13
    )
    plt.show()
    return fig, axes


def plot_rows_attack_vs_train_meanN(
    pt_attack_list,
    pt_train: str,
    N: int,
    t_idx: int = -1,
    wrap_title_width: int = 80,
    cmap_attack: str = "viridis",
    cmap_diff: str = "coolwarm",
    figsize_per_cell: float = 3.2,
    use_log: bool = False,
    log_eps: float = 1e-12,
    center_spectrum: bool = True,
):
    """
    每个 attack 一行、8 列：左 4 共享色条 + 右 4 各自色条
    每行四幅：FFT(x)、FFT(y_k)、FFT(x−a)、FFT(y_k−u_k)，再各自复制一套到右半区加独立色条。
    """
    assert isinstance(pt_attack_list, (list, tuple)) and len(pt_attack_list) > 0, \
        "pt_attack_list must be a non-empty list"

    nrows, ncols = len(pt_attack_list), 8
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_cell*ncols, figsize_per_cell*nrows),
        constrained_layout=True
    )
    if nrows == 1:
        axes = np.expand_dims(axes, 0)

    tag = "FFT(log10)" if use_log else "FFT"

    dt = _load_pt(pt_train)
    a_org_all, u_org_all = dt["x"], dt["y"]
    info_org = _parse_info_from_name(pt_train)

    for r, pta in enumerate(pt_attack_list):
        da = _load_pt(pta)
        x_att, y_att = da["x"], da["y"]
        if x_att.ndim != 3 or y_att.ndim != 4 or a_org_all.ndim != 3 or u_org_all.ndim != 4:
            raise ValueError("shapes must be x/a:(N,H,W), y/u:(N,H,W,T)")

        T = min(y_att.shape[3], u_org_all.shape[3])
        N_use = min(N, x_att.shape[0], y_att.shape[0], a_org_all.shape[0], u_org_all.shape[0])
        k = (t_idx % T)

        # FFT over (H,W) & complex-mean
        Xf = torch.fft.fft2(x_att[:N_use], dim=(-2, -1))
        Af = torch.fft.fft2(a_org_all[:N_use], dim=(-2, -1))
        Yf = torch.fft.fft2(y_att[:N_use, :, :, :T], dim=(1, 2))
        Uf = torch.fft.fft2(u_org_all[:N_use, :, :, :T], dim=(1, 2))
        if center_spectrum:
            Xf = torch.fft.fftshift(Xf, dim=(-2, -1)); Af = torch.fft.fftshift(Af, dim=(-2, -1))
            Yf = torch.fft.fftshift(Yf, dim=(1, 2));   Uf = torch.fft.fftshift(Uf, dim=(1, 2))
        Xm, Am = Xf.mean(0), Af.mean(0)           # (H,W)
        Ym, Um = Yf.mean(0), Uf.mean(0)           # (H,W,T)
        Yk, Uk = Ym[..., k], Um[..., k]           # (H,W)

        Dxa, Dyu = (Xm - Am), (Yk - Uk)

        def _disp(Z):
            A = torch.abs(Z).detach().cpu().numpy()
            if use_log:
                A = np.log10(A + float(log_eps))
            return A

        x_np = _disp(Xm); y_np = _disp(Yk)
        xd   = _disp(Dxa); yd   = _disp(Dyu)

        vmin_att = float(min(x_np.min(), y_np.min())); vmax_att = float(max(x_np.max(), y_np.max()))
        vmin_diff = float(min(xd.min(),   yd.min()));   vmax_diff = float(max(xd.max(),   yd.max()))

        row_axes = axes[r]

        # 左 4：共享色条
        im0 = row_axes[0].imshow(x_np, cmap=cmap_attack, vmin=vmin_att, vmax=vmax_att); row_axes[0].axis("off")
        row_axes[0].set_title(_wrap(f"{tag}: attack x\n(frame 1)", 20))
        im1 = row_axes[1].imshow(y_np, cmap=cmap_attack, vmin=vmin_att, vmax=vmax_att); row_axes[1].axis("off")
        row_axes[1].set_title(_wrap(f"{tag}: attack y\n(frame {k+1})", 20))
        im2 = row_axes[2].imshow(xd,   cmap=cmap_diff,   vmin=vmin_diff, vmax=vmax_diff); row_axes[2].axis("off")
        row_axes[2].set_title(_wrap(f"{tag}: diff x−a", 20))
        im3 = row_axes[3].imshow(yd,   cmap=cmap_diff,   vmin=vmin_diff, vmax=vmax_diff); row_axes[3].axis("off")
        row_axes[3].set_title(_wrap(f"{tag}: diff y−u", 20))

        fig.colorbar(im1, ax=row_axes[0:2].tolist(), fraction=0.04, pad=0.02)
        fig.colorbar(im3, ax=row_axes[2:4].tolist(), fraction=0.04, pad=0.02)

        # 右 4：单独色条
        im4 = row_axes[4].imshow(x_np, cmap=cmap_attack); row_axes[4].axis("off")
        row_axes[4].set_title(_wrap(f"{tag}: attack x\n(frame 1)", 20))
        fig.colorbar(im4, ax=row_axes[4], fraction=0.046, pad=0.04)

        im5 = row_axes[5].imshow(y_np, cmap=cmap_attack); row_axes[5].axis("off")
        row_axes[5].set_title(_wrap(f"{tag}: attack y\n(frame {k+1})", 20))
        fig.colorbar(im5, ax=row_axes[5], fraction=0.046, pad=0.04)

        im6 = row_axes[6].imshow(xd,   cmap=cmap_diff);  row_axes[6].axis("off")
        row_axes[6].set_title(_wrap(f"{tag}: diff x−a", 20))
        fig.colorbar(im6, ax=row_axes[6], fraction=0.046, pad=0.04)

        im7 = row_axes[7].imshow(yd,   cmap=cmap_diff);  row_axes[7].axis("off")
        row_axes[7].set_title(_wrap(f"{tag}: diff y−u", 20))
        fig.colorbar(im7, ax=row_axes[7], fraction=0.046, pad=0.04)

        desc_att = _desc_from_info(_parse_info_from_name(pta))
        row_axes[0].set_ylabel(
            _wrap(f"{tag} | attack[{desc_att}]\nmean of first N<= {N} (actual {N_use}), k={k+1}", wrap_title_width),
            rotation=0, ha="right", va="center", fontsize=10
        )
        row_axes[0].yaxis.set_label_coords(-0.12, 0.5)

    desc_org = _desc_from_info(info_org)
    fig.suptitle(
        _wrap(f"{tag} | Attack(mean of first N<= {N}) vs Train | train[{desc_org}]", 120),
        y=0.995, fontsize=12
    )
    fig.subplots_adjust(top=0.93)
    plt.show()
    return fig, axes


# ========== 一次读取、内部循环画：高效批量版本 ==========

def _precompute_fft_means_pair(pt_attack: str, pt_original: str, N: int, center_spectrum: bool = True):
    """
    读取 attack/original 两个 .pt（只读一次），返回打包 dict：
    - Xm, Ym: attack 的 FFT 复均值（x: (H,W), y: (H,W,T)）
    - Am, Um: original 的 FFT 复均值（a: (H,W), u: (H,W,T)）
    - T, t_x, t_frames, info_att, info_org, N_use
    """
    da = _load_pt(pt_attack)
    do = _load_pt(pt_original)
    x_att, y_att = da["x"], da["y"]    # (N,H,W), (N,H,W,T)
    x_org, y_org = do["x"], do["y"]    # (N,H,W), (N,H,W,T)

    if x_att.ndim != 3 or x_org.ndim != 3:
        raise ValueError(f"x must be (N,H,W). Got {tuple(x_att.shape)} and {tuple(x_org.shape)}")
    if y_att.ndim != 4 or y_org.ndim != 4:
        raise ValueError(f"y must be (N,H,W,T). Got {tuple(y_att.shape)} and {tuple(y_org.shape)}")

    # 统一 N & T
    N_use = min(N, x_att.shape[0], x_org.shape[0], y_att.shape[0], y_org.shape[0])
    T = min(y_att.shape[3], y_org.shape[3])

    # 空间 FFT -> 样本维复均值
    Xf = torch.fft.fft2(x_att[:N_use], dim=(-2, -1))
    Af = torch.fft.fft2(x_org[:N_use], dim=(-2, -1))
    Yf = torch.fft.fft2(y_att[:N_use, :, :, :T], dim=(1, 2))
    Uf = torch.fft.fft2(y_org[:N_use, :, :, :T], dim=(1, 2))
    if center_spectrum:
        Xf = torch.fft.fftshift(Xf, dim=(-2, -1))
        Af = torch.fft.fftshift(Af, dim=(-2, -1))
        Yf = torch.fft.fftshift(Yf, dim=(1, 2))
        Uf = torch.fft.fftshift(Uf, dim=(1, 2))

    Xm, Am = Xf.mean(0), Af.mean(0)     # (H,W)
    Ym, Um = Yf.mean(0), Uf.mean(0)     # (H,W,T)

    # 时间 & 元信息
    info_att = _parse_info_from_name(pt_attack)
    info_org = _parse_info_from_name(pt_original)
    t_final_str = info_att.get("t_final") or info_org.get("t_final") or ""
    t_x, t_frames = _infer_times(T, t_final_str)

    pack = dict(
        Xm=Xm, Ym=Ym, Am=Am, Um=Um,
        T=T, t_x=t_x, t_frames=t_frames,
        info_att=info_att, info_org=info_org, N_use=N_use
    )
    return pack

def _to_display(Z: torch.Tensor, use_log: bool, log_eps: float) -> np.ndarray:
    A = torch.abs(Z).detach().cpu().numpy()
    if use_log:
        A = np.log10(A + float(log_eps))
    return A

def plot_xy_and_frames_variants_once(
    pt_attack: str, pt_original: str,
    N: int, t_idx: int,
    modes=("shared", "separate"),
    logs=(True, False),
    grid=(5, 5),
    cmap_attack="viridis",
    cmap_diff="coolwarm",
    log_eps: float = 1e-12,
    center_spectrum: bool = True,
):
    """
    一次性 torch.load 两个数据，然后：
      for mode in modes:
        for use_log in logs:
          画 1×4 单图
          画 5×10 多帧图
    """
    pack = _precompute_fft_means_pair(pt_attack, pt_original, N, center_spectrum=center_spectrum)
    Xm, Ym, Am, Um = pack["Xm"], pack["Ym"], pack["Am"], pack["Um"]
    T, t_x, t_frames = pack["T"], pack["t_x"], pack["t_frames"]
    info_att, info_org, N_use = pack["info_att"], pack["info_org"], pack["N_use"]

    k = (t_idx % T)
    Yk, Uk = Ym[..., k], Um[..., k]
    Dxa = Xm - Am
    Dyu = Yk - Uk

    for mode in modes:
        for use_log in logs:
            tag = "FFT(log10)" if use_log else "FFT"

            # ---------- (A) 单图 1×4 ----------
            x_np = _to_display(Xm, use_log, log_eps)
            y_np = _to_display(Yk, use_log, log_eps)
            xd   = _to_display(Dxa, use_log, log_eps)
            yd   = _to_display(Dyu, use_log, log_eps)

            if mode == "shared":
                vmin_att = min(float(x_np.min()), float(y_np.min()))
                vmax_att = max(float(x_np.max()), float(y_np.max()))
                vmin_diff = min(float(xd.min()), float(yd.min()))
                vmax_diff = max(float(xd.max()), float(yd.max()))
            else:
                vmin_att = vmax_att = vmin_diff = vmax_diff = None

            fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)

            axes[0].imshow(x_np, cmap=cmap_attack, vmin=vmin_att, vmax=vmax_att); axes[0].axis("off")
            axes[0].set_title(f"{tag}: attack x\n(frame 1, t={_sec_str(t_x)})")

            t_k = t_frames[k] if t_frames is not None else None
            axes[1].imshow(y_np, cmap=cmap_attack, vmin=vmin_att, vmax=vmax_att); axes[1].axis("off")
            axes[1].set_title(f"{tag}: attack y\n(frame {k+1}, t={_sec_str(t_k)})")

            axes[2].imshow(xd, cmap=cmap_diff, vmin=vmin_diff, vmax=vmax_diff); axes[2].axis("off")
            axes[2].set_title(f"{tag}: diff x−a")

            axes[3].imshow(yd, cmap=cmap_diff, vmin=vmin_diff, vmax=vmax_diff); axes[3].axis("off")
            axes[3].set_title(f"{tag}: diff y−u")

            if mode == "shared":
                plt.colorbar(axes[1].images[0], ax=axes[:2].tolist(), fraction=0.04, pad=0.02)
                plt.colorbar(axes[3].images[0], ax=axes[2:].tolist(), fraction=0.04, pad=0.02)
            else:
                for ax in axes:
                    plt.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.04)

            desc_att = ", ".join([f"{kk}={vv}" for kk, vv in info_att.items() if vv])
            desc_org = ", ".join([f"{kk}={vv}" for kk, vv in info_org.items() if vv])
            fig.suptitle(
                f"Mean over first N={N_use} | {tag} | mode={mode} | attack[{desc_att}] | original[{desc_org}]",
                y=1.06, fontsize=13
            )
            plt.show()

            # ---------- (B) 多帧 5×10 ----------
            rows, cols = grid
            K = rows * cols
            num_to_plot = min(T, K)

            y_p = _to_display(Ym[..., :num_to_plot], use_log, log_eps)
            y_d = _to_display((Ym - Um)[..., :num_to_plot], use_log, log_eps)

            if mode == "shared":
                vmin_left,  vmax_left  = float(np.min(y_p)), float(np.max(y_p))
                vmin_right, vmax_right = float(np.min(y_d)), float(np.max(y_d))
            else:
                vmin_left = vmax_left = vmin_right = vmax_right = None

            fig2, axes2 = plt.subplots(rows, 2*cols, figsize=(2*cols*2.2, rows*2.2), constrained_layout=True)
            axes2 = np.asarray(axes2)

            ims_left, ims_right = [], []

            for kk in range(K):
                r, c = divmod(kk, cols)
                ax = axes2[r, c]
                if kk < num_to_plot:
                    if mode == "shared":
                        im = ax.imshow(y_p[..., kk], cmap=cmap_attack, vmin=vmin_left, vmax=vmax_left)
                    else:
                        im = ax.imshow(y_p[..., kk], cmap=cmap_attack)
                    t_k = (t_frames[kk] if (t_frames is not None and kk < len(t_frames)) else None)
                    ax.set_title(f"{tag}: attack y(k={kk+1})\n" + (f"t={_sec_str(t_k)}" if t_k is not None else ""), fontsize=9)
                    ims_left.append(im)
                ax.axis("off")

            for kk in range(K):
                r, c = divmod(kk, cols)
                ax = axes2[r, cols + c]
                if kk < num_to_plot:
                    if mode == "shared":
                        im = ax.imshow(y_d[..., kk], cmap=cmap_diff, vmin=vmin_right, vmax=vmax_right)
                    else:
                        im = ax.imshow(y_d[..., kk], cmap=cmap_diff)
                    t_k = (t_frames[kk] if (t_frames is not None and kk < len(t_frames)) else None)
                    ax.set_title(f"{tag}: diff y(k) − u(k)\n" + (f"t={_sec_str(t_k)}" if t_k is not None else ""), fontsize=9)
                    ims_right.append(im)
                ax.axis("off")

            if mode == "shared":
                if ims_left:  fig2.colorbar(ims_left[-1],  ax=axes2[:, :cols].ravel().tolist(), fraction=0.02, pad=0.01)
                if ims_right: fig2.colorbar(ims_right[-1], ax=axes2[:, cols:].ravel().tolist(), fraction=0.02, pad=0.01)
            else:
                for kk in range(num_to_plot):
                    r, c = divmod(kk, cols)
                    if axes2[r, c].images:
                        fig2.colorbar(axes2[r, c].images[0], ax=axes2[r, c], fraction=0.036, pad=0.01)
                    if axes2[r, cols + c].images:
                        fig2.colorbar(axes2[r, cols + c].images[0], ax=axes2[r, cols + c], fraction=0.036, pad=0.01)

            fig2.suptitle(
                f"Mean over first N={N_use} | {tag} amplitude | mode={mode} | attack y(k) vs diff(y(k)−u(k))",
                y=1.04, fontsize=13
            )
            plt.show()


def plot_rows_attack_vs_train_meanN_multi_once(
    pt_attack_list, pt_train: str, N: int, t_idx: int,
    logs=(True, False),
    cmap_attack="viridis", cmap_diff="coolwarm",
    figsize_per_cell=3.2,
    log_eps: float = 1e-12,
    center_spectrum: bool = True,
):
    """
    - train 只读一次；
    - 每个 attack 只读一次；
    - 对每个 use_log 版本各出一张多行 8 图。
    """
    # 先把 train 读好（保留原始张量，按每个 attack 的 N_use 再均值）
    dt = _load_pt(pt_train)
    a_all, u_all = dt["x"], dt["y"]     # (N,H,W), (N,H,W,T)
    if a_all.ndim != 3 or u_all.ndim != 4:
        raise ValueError("train shapes must be x:(N,H,W), y:(N,H,W,T)")

    info_org = _parse_info_from_name(pt_train)

    figs = []
    for use_log in logs:
        tag = "FFT(log10)" if use_log else "FFT"

        # 创建整张图
        nrows, ncols = len(pt_attack_list), 8
        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(figsize_per_cell*ncols, figsize_per_cell*nrows),
            constrained_layout=True
        )
        if nrows == 1:
            axes = np.expand_dims(axes, 0)

        for r, pta in enumerate(pt_attack_list):
            da = _load_pt(pta)
            x_att, y_att = da["x"], da["y"]
            if x_att.ndim != 3 or y_att.ndim != 4:
                raise ValueError("attack shapes must be x:(N,H,W), y:(N,H,W,T)")

            # 针对该 attack 计算 N_use & T，并用 train 的前 N_use 来对齐平均
            N_use = min(N, x_att.shape[0], y_att.shape[0], a_all.shape[0], u_all.shape[0])
            T = min(y_att.shape[3], u_all.shape[3])
            k = (t_idx % T)

            # 该 attack 的 FFT 复均值
            Xf = torch.fft.fft2(x_att[:N_use], dim=(-2, -1))
            Yf = torch.fft.fft2(y_att[:N_use, :, :, :T], dim=(1, 2))
            Af = torch.fft.fft2(a_all[:N_use], dim=(-2, -1))
            Uf = torch.fft.fft2(u_all[:N_use, :, :, :T], dim=(1, 2))
            if center_spectrum:
                Xf = torch.fft.fftshift(Xf, dim=(-2, -1)); Af = torch.fft.fftshift(Af, dim=(-2, -1))
                Yf = torch.fft.fftshift(Yf, dim=(1, 2));   Uf = torch.fft.fftshift(Uf, dim=(1, 2))

            Xm, Am = Xf.mean(0), Af.mean(0)       # (H,W)
            Ym, Um = Yf.mean(0), Uf.mean(0)       # (H,W,T)
            Yk, Uk = Ym[..., k], Um[..., k]

            Dxa, Dyu = (Xm - Am), (Yk - Uk)

            def _disp(Z):
                A = torch.abs(Z).detach().cpu().numpy()
                if use_log:
                    A = np.log10(A + float(log_eps))
                return A

            x_np = _disp(Xm); y_np = _disp(Yk)
            xd   = _disp(Dxa); yd   = _disp(Dyu)

            vmin_att = float(min(x_np.min(), y_np.min())); vmax_att = float(max(x_np.max(), y_np.max()))
            vmin_diff = float(min(xd.min(),   yd.min()));   vmax_diff = float(max(xd.max(),   yd.max()))

            row_axes = axes[r]
            im0 = row_axes[0].imshow(x_np, cmap=cmap_attack, vmin=vmin_att, vmax=vmax_att); row_axes[0].axis("off")
            row_axes[0].set_title(_wrap(f"{tag}: attack x\n(frame 1)", 20))
            im1 = row_axes[1].imshow(y_np, cmap=cmap_attack, vmin=vmin_att, vmax=vmax_att); row_axes[1].axis("off")
            row_axes[1].set_title(_wrap(f"{tag}: attack y\n(frame {k+1})", 20))
            im2 = row_axes[2].imshow(xd,   cmap=cmap_diff,   vmin=vmin_diff, vmax=vmax_diff); row_axes[2].axis("off")
            row_axes[2].set_title(_wrap(f"{tag}: diff x−a", 20))
            im3 = row_axes[3].imshow(yd,   cmap=cmap_diff,   vmin=vmin_diff, vmax=vmax_diff); row_axes[3].axis("off")
            row_axes[3].set_title(_wrap(f"{tag}: diff y−u", 20))

            fig.colorbar(im1, ax=row_axes[0:2].tolist(), fraction=0.04, pad=0.02)
            fig.colorbar(im3, ax=row_axes[2:4].tolist(), fraction=0.04, pad=0.02)

            im4 = row_axes[4].imshow(x_np, cmap=cmap_attack); row_axes[4].axis("off")
            row_axes[4].set_title(_wrap(f"{tag}: attack x\n(frame 1)", 20))
            fig.colorbar(im4, ax=row_axes[4], fraction=0.046, pad=0.04)

            im5 = row_axes[5].imshow(y_np, cmap=cmap_attack); row_axes[5].axis("off")
            row_axes[5].set_title(_wrap(f"{tag}: attack y\n(frame {k+1})", 20))
            fig.colorbar(im5, ax=row_axes[5], fraction=0.046, pad=0.04)

            im6 = row_axes[6].imshow(xd,   cmap=cmap_diff);  row_axes[6].axis("off")
            row_axes[6].set_title(_wrap(f"{tag}: diff x−a", 20))
            fig.colorbar(im6, ax=row_axes[6], fraction=0.046, pad=0.04)

            im7 = row_axes[7].imshow(yd,   cmap=cmap_diff);  row_axes[7].axis("off")
            row_axes[7].set_title(_wrap(f"{tag}: diff y−u", 20))
            fig.colorbar(im7, ax=row_axes[7], fraction=0.046, pad=0.04)

            desc_att = _desc_from_info(_parse_info_from_name(pta))
            row_axes[0].set_ylabel(
                _wrap(f"{tag} | attack[{desc_att}]\nmean of first N<= {N} (actual {N_use}), k={k+1}", 80),
                rotation=0, ha="right", va="center", fontsize=10
            )
            row_axes[0].yaxis.set_label_coords(-0.12, 0.5)

        desc_org = ", ".join([f"{k}={v}" for k, v in info_org.items() if v])
        fig.suptitle(
            _wrap(f"{tag} | Attack(mean of first N<= {N}) vs Train | train[{desc_org}]", 120),
            y=0.995, fontsize=12
        )
        fig.subplots_adjust(top=0.93)
        plt.show()
        figs.append(fig)

    return figs


# ========== 示例调用（按需修改路径） ==========

if __name__ == "__main__":
    # ---- 单个 attack 与单个 original（一次读取，循环画所有版本）----
    pt_attack = "dim2d_nx256_N500_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha50_epsilon39.3216_steps10.pt"
    pt_orig   = "../../../exponax_datasets/t20/dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames.pt"

    # 只读一次，在内部把 (cbar_mode, use_log) 全部组合画完（含单图 + 多帧）
    plot_xy_and_frames_variants_once(
        pt_attack, pt_orig,
        N=50, t_idx=19,
        modes=("shared","separate"),   # 你也可以传 ("shared",) 只画共享色条版本
        logs=(True, False),            # 你也可以传 (True,) 只画 log 版本
        grid=(5,5),
        center_spectrum=True
    )

    # ---- 多个 attack 与同一个 original（train 只读一次，attack 各读一次）----
    pt_attack_list = [
        "dim2d_nx256_N500_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha10_epsilon39.3216_steps10.pt",
        "dim2d_nx256_N500_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha50_epsilon39.3216_steps10.pt",
    ]
    plot_rows_attack_vs_train_meanN_multi_once(
        pt_attack_list, pt_orig,
        N=500, t_idx=-2,
        logs=(True, False),
        center_spectrum=True,
        figsize_per_cell=3.0
    )

    # 如果你仍要用“基础绘图”逐个调用（会各自读取），也可以：
    # plot_xy_compare_attack_original_avg(pt_attack, pt_orig, N=50, t_idx=19, cbar_mode="shared", use_log=False)
    # plot_frames_y_and_diff_side_by_side_avg(pt_attack, pt_orig, N=50, grid=(5,5), cbar_mode="shared", use_log=True)
    # plot_rows_attack_vs_train_meanN(pt_attack_list, pt_orig, N=500, t_idx=-2, use_log=False, figsize_per_cell=3.0)

AttributeError: module 'torch' has no attribute '_utils'

In [1]:
import torch

data = torch.load("dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_all_frames_modespec=wwwwwwwwww_norm2_alpha50_epsilon78.6432_steps10.pt", weights_only=False)

KeyboardInterrupt: 